# Monte Carlo Validation

Validate the Metropolis-Hastings sampler for scalar φ⁴ theory against known results:
1. Free field (λ=0): Gaussian distribution with known variance
2. Phase transition: order parameter vs coupling
3. Two-point correlation function decay

In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
    !pip install -q torch-geometric omegaconf
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
    os.chdir(PROJECT_ROOT)
else:
    sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from qft_graph.config import LatticeConfig, ScalarFieldConfig, MCConfig
from qft_graph.lattice.hypercubic import HypercubicLattice
from qft_graph.actions.phi4 import Phi4Action
from qft_graph.mc.metropolis import MetropolisSampler
from qft_graph.mc.observables import ObservableSet
from qft_graph.mc.analysis import jackknife_mean_error
from qft_graph.utils.reproducibility import set_seed

set_seed(42)
plt.style.use('dark_background')
%matplotlib inline

## 1. Free Field Validation (λ=0)

In [ ]:
lattice = HypercubicLattice(LatticeConfig(dimensions=(16, 16)))
action = Phi4Action(lattice, ScalarFieldConfig(mass_squared=1.0, coupling=0.0))
sampler = MetropolisSampler(action, MCConfig(n_configs=2000, n_thermalization=500, n_sweeps_between=10, step_size=0.7, seed=42))

result = sampler.generate(2000)
print(f'Acceptance rate: {result.acceptance_rate:.3f}')
print(f'Mean action: {result.actions.mean():.2f} +/- {result.actions.std():.2f}')

# Histogram of field values (should be Gaussian)
fig, ax = plt.subplots(figsize=(8, 5))
all_vals = result.configurations.flatten().numpy()
ax.hist(all_vals, bins=100, density=True, alpha=0.7, label='MC samples')
x = np.linspace(-3, 3, 200)
ax.set_xlabel(r'$\phi$')
ax.set_ylabel('Density')
ax.set_title(r'Free field ($\lambda=0$, $m^2=1$): should be Gaussian')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Phase Transition: Order Parameter vs m²

In [ ]:
# Scan from disordered (m²=0) to deep ordered (m²=-2.5) with warm-starting
# Track |M|, <M²>, and susceptibility χ = V·(<M²> - <|M|>²)
m2_values = np.linspace(0.0, -2.5, 30)
mags, mag_errs = [], []
m2_vals_list, m2_errs = [], []
chis = []

V = lattice.num_sites()  # 256 for 16x16
warm_phi = None

for m2 in m2_values:
    action = Phi4Action(lattice, ScalarFieldConfig(mass_squared=m2, coupling=0.5))
    mc_cfg = MCConfig(
        n_configs=1000,
        n_thermalization=1000,
        n_sweeps_between=10,
        seed=None,
    )
    sampler = MetropolisSampler(action, mc_cfg)
    result = sampler.generate(mc_cfg.n_configs, initial_phi=warm_phi)
    warm_phi = result.configurations[-1].clone()

    n = len(result.configurations)

    # Per-config magnetization: M_i = (1/V) Σ_x φ(x)
    M_samples = torch.tensor([
        result.configurations[i].mean().item() for i in range(n)
    ])
    absM_samples = M_samples.abs()
    M2_samples = M_samples ** 2

    # |M| with jackknife errors
    absM_mean, absM_err = jackknife_mean_error(absM_samples)
    mags.append(absM_mean)
    mag_errs.append(absM_err)

    # <M²> with jackknife errors
    m2_mean, m2_err = jackknife_mean_error(M2_samples)
    m2_vals_list.append(m2_mean)
    m2_errs.append(m2_err)

    # Susceptibility χ = V · (<M²> - <|M|>²)
    chi = V * (M2_samples.mean().item() - absM_samples.mean().item()**2)
    chis.append(chi)

    print(f'm²={m2:.2f}: |M|={absM_mean:.4f} ± {absM_err:.4f}, '
          f'<M²>={m2_mean:.5f}, χ={chi:.2f}, acc={result.acceptance_rate:.3f}')

# --- Three-panel plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: |M| (order parameter)
axes[0].errorbar(m2_values, mags, yerr=mag_errs, fmt='o-', capsize=3)
axes[0].axhline(y=1/np.sqrt(V), color='red', ls=':', alpha=0.5, label=r'$1/\sqrt{V}$ noise floor')
axes[0].set_xlabel(r'$m^2$')
axes[0].set_ylabel(r'$|\langle\phi\rangle|$')
axes[0].set_title(r'$|\langle M \rangle|$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Panel 2: <M²> (doesn't cancel between ±M sectors)
axes[1].errorbar(m2_values, m2_vals_list, yerr=m2_errs, fmt='s-', capsize=3, color='orange')
axes[1].set_xlabel(r'$m^2$')
axes[1].set_ylabel(r'$\langle M^2 \rangle$')
axes[1].set_title(r'$\langle M^2 \rangle$')
axes[1].grid(True, alpha=0.3)

# Panel 3: Susceptibility (peaks at critical point)
axes[2].plot(m2_values, chis, 'D-', color='lime')
axes[2].set_xlabel(r'$m^2$')
axes[2].set_ylabel(r'$\chi$')
axes[2].set_title(r'Susceptibility $\chi = V(\langle M^2\rangle - \langle|M|\rangle^2)$')
axes[2].grid(True, alpha=0.3)

fig.suptitle(r'Phase transition scan ($\lambda=0.5$, $L=16$)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Two-Point Correlation Function

In [ ]:
# Two-point correlation at multiple m² values: disordered, near-critical, ordered
# Uses connected correlator and both ξ estimators (log-slope + second-moment)
L = lattice.shape[0]
m2_probe = [-0.5, -1.5, -1.9, -2.1]
colors = ['cyan', 'orange', 'lime', 'magenta']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for m2, color in zip(m2_probe, colors):
    action = Phi4Action(lattice, ScalarFieldConfig(mass_squared=m2, coupling=0.5))
    sampler = MetropolisSampler(action, MCConfig(
        n_configs=1000, n_thermalization=1000, n_sweeps_between=10, seed=42
    ))
    result = sampler.generate(1000)
    obs = ObservableSet(lattice)

    # Average connected G(r) over configurations
    G_r_avg = torch.zeros(L // 2 + 1)
    n_avg = min(500, len(result.configurations))
    for i in range(n_avg):
        G_r_avg += obs.two_point_function(result.configurations[i], connected=True)
    G_r_avg /= n_avg

    xi_log = ObservableSet.correlation_length(G_r_avg)
    xi_sm = ObservableSet.correlation_length_second_moment(G_r_avg, L)
    r = np.arange(len(G_r_avg))

    # Left panel: G(r) with log-slope fit
    axes[0].semilogy(r, G_r_avg.abs().numpy(), 'o-', color=color,
                     label=rf'$m^2={m2}$, $\xi_{{log}}={xi_log:.2f}$')
    if 0 < xi_log < 50:
        fit_r = np.linspace(1, len(G_r_avg)-1, 100)
        axes[0].semilogy(fit_r, G_r_avg[1].abs().item() * np.exp(-fit_r/xi_log),
                         '--', color=color, alpha=0.4)

    # Right panel: G(r) with second-moment ξ
    axes[1].semilogy(r, G_r_avg.abs().numpy(), 'o-', color=color,
                     label=rf'$m^2={m2}$, $\xi_{{SM}}={xi_sm:.2f}$')
    if 0 < xi_sm < 50:
        fit_r = np.linspace(1, len(G_r_avg)-1, 100)
        axes[1].semilogy(fit_r, G_r_avg[1].abs().item() * np.exp(-fit_r/xi_sm),
                         '--', color=color, alpha=0.4)

    print(f'm²={m2}: ξ_log={xi_log:.3f}, ξ_SM={xi_sm:.3f}, '
          f'ξ_SM/L={xi_sm/L:.4f}, acc={result.acceptance_rate:.3f}')

for ax in axes:
    ax.set_xlabel(r'$r/a$')
    ax.set_ylabel(r'$|G_c(r)|$ (connected)')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_title(r'Log-slope $\xi$ estimator')
axes[1].set_title(r'Second-moment $\xi$ estimator (robust)')

fig.suptitle(r'Connected two-point function ($\lambda=0.5$, $L=16$)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()